In [9]:
import numpy as np
import matplotlib.pyplot as plt
from matplotlib.ticker import ScalarFormatter
import pandas as pd
import openflash as of
from scipy.special import hankel1e 
import capytaine as cpt

In [3]:
rho=1000
g=9.81
omega_sweep = np.linspace(0.5,2.0,10)
h = 1.001            # Water Depth (m)
d_list = [0.5, 0.25]   # Step depths (m) for inner and outer bodies
a_list = [0.5, 1.0]    # Radii (m) for inner and outer bodies
NMK = [30, 30, 30]     # Harmonics for inner, middle, and exterior domains
heaving_list = True 

In [8]:
# m0 = of.multi_equations.wavenumber(omega_sweep[j], h)

bodies_sweep = []

# Single Body
body = of.SteppedBody(
    a=np.array(a_list),
    d=np.array(d_list),
    slant_angle= np.array([0, 0]),
    heaving=heaving_list
)

bodies_sweep.append(body)

# 2. Create arrangement
arrangement_sweep = of.ConcentricBodyGroup(bodies_sweep)

# 3. Create geometry
geometry_sweep = of.BasicRegionGeometry(
    body_arrangement=arrangement_sweep,
    h=h,
    NMK=NMK
)

# 4. Create the MEEMProblem instance
problem = of.MEEMProblem(geometry_sweep)

# 5. Set the frequencies for the sweep
problem.set_frequencies(omega_sweep)

# 6. Initialize a new MEEM Engine for this problem
engine = of.MEEMEngine(problem_list=[problem])

results = engine.run_and_store_results(0)
print(results.get_results())
# for omega in omega_sweep:
#     m0 = of.multi_equations.wavenumber(omega, h)
#     X = engine.solve_linear_system_multi(problem, m0)
#     coeffs = engine.compute_hydrodynamic_coefficients(problem, X, m0)




Hydrodynamic coefficients stored in xarray dataset.
Potentials stored in xarray dataset (batched across frequencies/modes).
<xarray.Dataset> Size: 48kB
Dimensions:             (frequency: 10, mode_i: 1, mode_j: 1,
                         c_vector_index: 90, harmonics: 60, domain_name: 3,
                         modes: 1)
Coordinates:
  * frequency           (frequency) float64 80B 0.5 0.6667 0.8333 ... 1.833 2.0
  * mode_i              (mode_i) int64 8B 0
  * mode_j              (mode_j) int64 8B 0
  * c_vector_index      (c_vector_index) int64 720B 0 1 2 3 4 ... 85 86 87 88 89
  * harmonics           (harmonics) int64 480B 0 1 2 3 4 5 ... 54 55 56 57 58 59
  * domain_name         (domain_name) int64 24B 0 1 2
  * modes               (modes) int64 8B 0
Data variables:
    added_mass          (frequency, mode_i, mode_j) float64 80B 4.109e+03 ......
    damping             (frequency, mode_i, mode_j) float64 80B 1.183e+03 ......
    excitation_force    (frequency, mode_i) float64 80B 3

In [ ]:
def MEEM_hydro_coeffs(h,omega_sweep,d_out,a_list,NMK,heaving_list):

    A_MEEM = np.zeros_like(omega_sweep)
    B_MEEM = np.zeros_like(omega_sweep)
    F_ex_MEEM = np.zeros(len(omega_sweep), dtype=np.complex128)

    for j in range(len(omega_sweep)):
        # Run MEEM
        d_list = d_out # drafts [m] 

        # 1. Create SteppedBody objects
        bodies_sweep = []

        m0 = of.multi_equations.wavenumber(omega_sweep[j], h)
        
        # Single Body
        body = of.SteppedBody(
            a=np.array(a_list),
            d=np.array(d_list),
            slant_angle= np.array([0, 0]),
            heaving=heaving_list
        )
        bodies_sweep.append(body)
        

        # 2. Create arrangement
        arrangement_sweep = of.ConcentricBodyGroup(bodies_sweep)

        # 3. Create geometry
        geometry_sweep = of.BasicRegionGeometry(
            body_arrangement=arrangement_sweep,
            h=h,
            NMK=NMK
        )

        # 4. Create the MEEMProblem instance
        problem = of.MEEMProblem(geometry_sweep)

        # 5. Set the frequencies for the sweep
        problem.set_frequencies(np.array([omega_sweep[j]]))

        # 6. Initialize a new MEEM Engine for this problem
        engine = of.MEEMEngine(problem_list=[problem])

        # 7. Solve
        X = engine.solve_linear_system_multi(problem, m0)

        # 8. Coefficients
        coeffs = engine.compute_hydrodynamic_coefficients(problem, X, m0)
        # print(coeffs[0])

        # excitation force magnituide equation
        lambda_0e = m0
        f0 = np.sinh(2*lambda_0e*h)
        N_0 = (1/2) * (1 + (f0/(2*lambda_0e*h)))
        F_ex_MEEM[j] = ((-4 * 1j * rho * g * h * np.sqrt(N_0) * X[-NMK[-1]])/
                        (np.cosh(lambda_0e * h) *  hankel1e(0,lambda_0e * a_list[-1]) * np.exp(1j * lambda_0e * a_list[-1])) 
                         )

        
        A_MEEM[j] = coeffs[0]['real']
        B_MEEM[j] = coeffs[0]['imag']
        # F_ex_MEEM[j] = coeffs[0]['excitation_force'] * np.exp(1j * coeffs[0]['excitation_phase'])

    return A_MEEM, B_MEEM, F_ex_MEEM